# CANDLE V3 Direct FC2 — full engineering baseline

Engineering smoke: 27/27 structural tests and 80/80 native checks passed.
One controlled synthetic install; zero natural installs in the bounded real-data smoke.
Transfer effectiveness and historical safety remain unproven. This is the first full run.

- Enable Internet and GPU on Kaggle; use a fresh session and empty OUTPUT_DIR.
- Attach `/kaggle/input/datasets/khoilv2005/100-clients`.
- Fresh seed 42, 100 clients, T0–T5, 20 rounds/task, batch 2048, one GPU with AMP.
- Native V1 paper neighborhoods, xi=0.8; one local model/router per client, binary_cosine.
- Direct FC2 transfer is enabled explicitly. No extra donor/head optimizer or fallback variant.
- Teaching runs at task end, bounded to one donor/target per receiver and 128 MiB per task.
- Uses the original V1 BASE/CAL/FIT/validation split; BASE is 80% of original TRAIN.
  Each task trains only its current BASE; no historical raw-data replay.
- Current FIT/CAL are used for transfer decisions. Missing evidence or rejected transfers
  do not stop native training; acceptance gates are unchanged.
- No intermediate TEST. After T5/R19 and terminal consolidation, evaluate full original
  34-class TEST on disjoint receiver shards: pooled BinarySelf primary, MulticlassSelf secondary.
- Round checkpoints are compressed and each task seals a ZIP; 12 GiB output guard.
  This fresh-run launcher does not expose arbitrary round resume.
- Source comes from the pinned GitHub revision; no embedded source, datasets or checkpoints.
  Existing mounted V1 roles can be reused via ROLES_DIR; otherwise roles are prepared first.

Results: `results_candle_v3_direct_fc2_xi_0p8_seed_42/` under Kaggle Output.
Completion: `candle_v3_completion.json`; final evaluation: `candle_v3_self_task_5.zip`.


In [ ]:
# CANDLE V3 Direct FC2: full engineering baseline; edit paths only.
import os
from pathlib import Path
import subprocess
import sys

CODE_REF = "88f76a2a986f47bacb924a4bf93fcd393d18d32d"
DATA_DIR = "/kaggle/input/datasets/khoilv2005/100-clients"
OUTPUT_DIR = "/kaggle/working/results_candle_v3_direct_fc2_xi_0p8_seed_42"
# Point to mounted historical role_lock.json's DIRECTORY to reuse the exact split.
# An empty directory here recreates roles from original TRAIN using split seed20261006.
ROLES_DIR = "/kaggle/working/candle_v3_roles"

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
code = Path("/kaggle/temp/candle_v3_direct_fc2_source")
code.parent.mkdir(parents=True, exist_ok=True)
if not code.exists():
    subprocess.check_call(["git", "clone", "--filter=blob:none", "--no-checkout",
        "https://github.com/khoilv2005/FL_IL_IDS.git", str(code)], env=env)
    subprocess.check_call(["git", "sparse-checkout", "set", "--no-cone",
        "/candle_v3_runtime/", "/tools/", "/configs/",
        "/requirements-candle-v3-kaggle.txt"], cwd=code, env=env)
    subprocess.check_call(["git", "checkout", "--detach", CODE_REF], cwd=code, env=env)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=code, text=True).strip()
if actual != CODE_REF:
    raise ValueError("Source differs from pinned CANDLE V3 revision; use a fresh session")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
    str(code / "requirements-candle-v3-kaggle.txt")])
print("CANDLE V3 supporting source:", actual, flush=True)
subprocess.check_call([sys.executable, "-u", str(code / "tools/train_candle_v3.py"),
    "--direct-fc2", "--data-dir", DATA_DIR, "--output", OUTPUT_DIR, "--roles", ROLES_DIR], cwd=code, env=env)
print(Path(OUTPUT_DIR, "candle_v3_completion.json").read_text(encoding="utf-8"))
